## nn.Module — Classification

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split


In [2]:
## Data

transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5, ))])
full_train = datasets.FashionMNIST(root="./data", train=True,  download=True, transform=transform)
test_data  = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

train_data, val_data = random_split(full_train, [48000, 12000])

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=256)
test_loader  = DataLoader(test_data,  batch_size=256)


100%|██████████| 26.4M/26.4M [00:10<00:00, 2.57MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 186kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.77MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 2.79MB/s]


## Model


In [3]:
class FashionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1  = nn.Linear(784, 256)
        self.drop = nn.Dropout(0.3)
        self.fc2  = nn.Linear(256, 128)
        self.out  = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 784)                      # flatten (B,1,28,28) → (B,784)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        x = F.relu(self.fc2(x))
        return self.out(x)  
    
    
model     = FashionClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


## Training loop

In [4]:
for epoch in range(20):
    model.train()
    for X_batch, y_batch in train_loader:
        logits=model(X_batch)
        loss=criterion(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    # Validation
    model.eval()
    correct=total=0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            preds=model(X_batch).argmax(dim=1)
            correct+=(preds==y_batch).sum().item()
            total+=y_batch.size(0)
            
    if (epoch+1)%5==0:
        print(f"Epoch {epoch+1:2d} | Val acc: {correct/total:.4f}")
        

Epoch  5 | Val acc: 0.8668
Epoch 10 | Val acc: 0.8860
Epoch 15 | Val acc: 0.8884
Epoch 20 | Val acc: 0.8937


In [5]:
# Test
model.eval()
correct=total=0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds=model(X_batch).argmax(dim=1)
        correct+=(preds==y_batch).sum().item()
        total+=y_batch.size(0)

print(f"Test accuracy:{correct/total:.4f}")

Test accuracy:0.8854
